# Step 4: The Final AI Root Cause Analyzer System
Putting it all together, this is our end-to-end RAG (Retrieval-Augmented Generation) pipeline for IT incidents.

In [1]:
import chromadb
import ollama
import json

CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "incident_logs"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2"

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(name=COLLECTION_NAME)

def get_embedding(text):
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return response["embedding"]

In [2]:
def analyze_incident(system_name, error_msg):
    print(f"Analyzing new incident in {system_name}...")
    
    # 1. Embed the new incident
    incident_text = f"\nSystem: {system_name}\nError: {error_msg}\n"
    query_embedding = get_embedding(incident_text)
    
    # 2. Retrieve similar past incidents from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=2
    )
    
    context = "\n".join(results['documents'][0]) if results['documents'] else "No similar historical incidents found."
    
    # 3. Generate RCA using an LLM
    prompt = f"""
    You are an AI IT Operations assistant.
    
    New Incident:
    {incident_text}
    
    Historical Context (Similar past incidents):
    {context}
    
    Using the historical context, please analyze the new incident.
    Provide:
    1. The Likely Root Cause
    2. A Proposed Fix
    
    Keep it structured and concise.
    """
    
    response = ollama.chat(model=LLM_MODEL, messages=[
        {'role': 'user', 'content': prompt}
    ])
    
    return response['message']['content']

Let's test our complete pipeline on a brand new 500 error from the frontend!

In [3]:
new_system = "frontend"
new_error = "API returned 500 Internal Server Error when clicking checkout"

final_report = analyze_incident(new_system, new_error)
print("\n=== FINAL RCA REPORT ===\n")
print(final_report)

Analyzing new incident in frontend...



=== FINAL RCA REPORT ===

**Incident Analysis**

**System:** frontend
**Error:** API returned 500 Internal Server Error when clicking checkout

**Historical Context:**

* Similar incident occurred in the past due to a null pointer exception, resolved by adding null checks.
* Another similar incident was related to a database connection timeout in the payment-service system, which was resolved by increasing the DB pool size.

**Likely Root Cause:**
Based on the historical context, it is likely that the null pointer exception is reappearing as the root cause of this new incident. The null pointer exception could be causing an error in the API response when checking out, resulting in a 500 Internal Server Error.

**Proposed Fix:**

1. Add null checks to ensure that all required parameters are populated before calling the API.
2. Review and validate the API endpoint parameters to prevent null values from being passed.

This fix should address the root cause of the issue and resolve the 50